## Setup

In [2]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from vocab import Vocab
from nmt_model import NMT
import dataset

import torch

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/iliarudiak/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
# Set up a logger for "nmt.dataset" level to DEBUG for this notebook
logger_dataset = logging.getLogger("nmt")
logger_dataset.setLevel(logging.DEBUG)

In [3]:
# Set up a logger for "nmt.model" level to DEBUG for this notebook
logger_model = logging.getLogger("nmt.model")
logger_model.setLevel(logging.DEBUG)

## 01 Dataset Debugging

`NMT` class performs the following operations:
- it reads a pair of source and target text files line by line.
- it tokenizes each line using pre-trained SentencePiece models.
- it converts the tokenized subwords into integer IDs using pre-built vocabularies.

`__getitem__` converts a pair of source and target IDs into PyTorch LongTensors.

In [26]:
# Set some parameters for the model
vocab_file = 'vocab.json'
embed_size = 64
hidden_size = 128
batch_size = 2

# Load the vocabulary from the specified path
vocab = Vocab.load(vocab_file)

# Create toy train dataset
toy_train_dataset = dataset.TranslationDataset(
    src_file_path='zh_en_data/train_1K.zh',
    tgt_file_path='zh_en_data/train_1K.en',
    src_vocab=vocab.src,
    tgt_vocab=vocab.tgt,
    src_sp_model='src.model',
    tgt_sp_model='tgt.model',
)

DEBUG:nmt.dataset:  ===line_idx=0===
DEBUG:nmt.dataset:  line_src=事发时身穿白色外衣,深色上衣及深色褲。

DEBUG:nmt.dataset:  line_tgt=he was seen wearing a white jacket, a dark-coloured t-shirt and pants.   

DEBUG:nmt.dataset:  src_pieces=['▁', '事发', '时', '身穿', '白色', '外', '衣', ',', '深', '色', '上', '衣', '及', '深', '色', '褲', '。']
DEBUG:nmt.dataset:  len=17 src_ids=[7, 8735, 33, 7695, 8367, 189, 3082, 4, 1072, 1517, 35, 3082, 17, 1072, 1517, 13275, 5]
DEBUG:nmt.dataset:  tgt_pieces=['<s>', '▁he', '▁was', '▁seen', '▁wear', 'ing', '▁a', '▁white', '▁', 'jacket', ',', '▁a', '▁dark', '-', 'coloured', '▁t', '-', 'shi', 'rt', '▁and', '▁p', 'ants', '.', '</s>']
DEBUG:nmt.dataset:  len=24 tgt_ids=[1, 49, 23, 891, 2750, 20, 12, 2258, 19, 7854, 7, 12, 3792, 13, 7992, 517, 13, 3622, 2331, 10, 427, 3221, 5, 2]
DEBUG:nmt.dataset:  ===line_idx=1===
DEBUG:nmt.dataset:  line_src=香港浸会大学体毓及康乐管理学亦向参加者解释运动与身体状湟的关系。

DEBUG:nmt.dataset:  line_tgt=they were also briefed by the physical education and recreation management society o

We may see that it returns: 
- `src_ids` of the line 0 above converted into a PyTorch LongTensor.
- `tgt_ids` of the line 0 above converted into a PyTorch LongTensor.

In [22]:
toy_train_dataset[0]

(tensor([    7,  8735,    33,  7695,  8367,   189,  3082,     4,  1072,  1517,
            35,  3082,    17,  1072,  1517, 13275,     5]),
 tensor([   1,   49,   23,  891, 2750,   20,   12, 2258,   19, 7854,    7,   12,
         3792,   13, 7992,  517,   13, 3622, 2331,   10,  427, 3221,    5,    2]))

Collate callable for batching TranslationDataset samples:
1. Sorts the batch by source sequence length in descending order (required for pack_padded_sequence).
2. Measures source lengths.
3. Pads source and target sequences to batch maximum lengths using pad_sequence. (dynamic padding)
4. Returns tensors shaped (src_len, batch_size) and (tgt_len, batch_size).

It returns `src_padded, src_lengths, tgt_padded`.

In [27]:
# Create toy train dataloader
toy_train_dataloader = dataset.get_dataloader(
    dataset=toy_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    pad_id=0,
    num_workers=0,
    pin_memory=False,
)

# Get a simple batch from the toy train dataloader
toy_train_batch = next(iter(toy_train_dataloader))
src_padded, src_lengths, tgt_padded = toy_train_batch

We may see that the `batch_size` corresponds to the second dimension. We may also see the dynamic padding to the maximum length of the sequences in the batch. We may also see that the sequences are sorted by length in descending order. The shorted sequence is padded up to the length of the longest sequence in the batch (dynamic padding), in our case the sequences of length 17 is padded with 2 tokens up to length 19.

In [29]:
src_padded.shape, src_lengths

(torch.Size([19, 2]), [19, 17])

In [30]:
src_padded

tensor([[  731,     7],
        [11949,  8735],
        [   39,    33],
        [  494,  7695],
        [ 2592,  8367],
        [   17,   189],
        [ 8894,  3082],
        [  237,     4],
        [  515,  1072],
        [  127,  1517],
        [   52,    35],
        [ 4597,  3082],
        [  788,    17],
        [  725,  1072],
        [   18,  1517],
        [ 2448, 13275],
        [ 7132,     5],
        [ 2054,     0],
        [    5,     0]])

## 02 Model Debugging 

### 01 Create a sample batch

In [5]:
# Set some parameters for the model
vocab_file = 'vocab.json'
embed_size = 64
hidden_size = 128
batch_size = 2

# Load the vocabulary from the specified path
vocab = Vocab.load(vocab_file)

# Create toy train dataset
toy_train_dataset = dataset.TranslationDataset(
    src_file_path='zh_en_data/train_1K.zh',
    tgt_file_path='zh_en_data/train_1K.en',
    src_vocab=vocab.src,
    tgt_vocab=vocab.tgt,
    src_sp_model='src.model',
    tgt_sp_model='tgt.model',
)

# Create toy train dataloader
toy_train_dataloader = dataset.get_dataloader(
    dataset=toy_train_dataset,
    batch_size=batch_size,
    shuffle=False,
    pad_id=0,
    num_workers=0,
    pin_memory=False,
)

# Get a simple batch from the toy train dataloader
toy_train_batch = next(iter(toy_train_dataloader))
src_padded, src_lengths, tgt_padded = toy_train_batch

DEBUG:nmt.dataset:  ===line_idx=0===
DEBUG:nmt.dataset:  line_src=事发时身穿白色外衣,深色上衣及深色褲。

DEBUG:nmt.dataset:  line_tgt=he was seen wearing a white jacket, a dark-coloured t-shirt and pants.   

DEBUG:nmt.dataset:  src_pieces=['▁', '事发', '时', '身穿', '白色', '外', '衣', ',', '深', '色', '上', '衣', '及', '深', '色', '褲', '。']
DEBUG:nmt.dataset:  len=17 src_ids=[7, 8735, 33, 7695, 8367, 189, 3082, 4, 1072, 1517, 35, 3082, 17, 1072, 1517, 13275, 5]
DEBUG:nmt.dataset:  tgt_pieces=['<s>', '▁he', '▁was', '▁seen', '▁wear', 'ing', '▁a', '▁white', '▁', 'jacket', ',', '▁a', '▁dark', '-', 'coloured', '▁t', '-', 'shi', 'rt', '▁and', '▁p', 'ants', '.', '</s>']
DEBUG:nmt.dataset:  len=24 tgt_ids=[1, 49, 23, 891, 2750, 20, 12, 2258, 19, 7854, 7, 12, 3792, 13, 7992, 517, 13, 3622, 2331, 10, 427, 3221, 5, 2]
DEBUG:nmt.dataset:  ===line_idx=1===
DEBUG:nmt.dataset:  line_src=香港浸会大学体毓及康乐管理学亦向参加者解释运动与身体状湟的关系。

DEBUG:nmt.dataset:  line_tgt=they were also briefed by the physical education and recreation management society o

Let's see if we change the length of `tgt_padded` in our collate function for teacher forcing. The length of `tgt_pieces` is `[24, 34]`. This includes BOS and EOS tokens. As we may see, the `tht_padded` has the same 34 length, so it does *not* remove any tokens for teacher forcing.

In [42]:
tgt_padded.shape

torch.Size([34, 2])

### 02 Forward pass: `encode` method

In [38]:
# Instantiate the model
model = NMT(embed_size=embed_size, 
            hidden_size=hidden_size, 
            vocab=vocab,
            dropout_rate=0.0)

# Forward pass
output = model(src_padded, src_lengths, tgt_padded)

DEBUG:nmt.model:  ===INIT NMT MODEL===
DEBUG:nmt.model:  Embed size: 64, Hidden size: 128, Dropout rate: 0.0
DEBUG:nmt.model:  Source vocab size: 21001, Target vocab size: 8001
DEBUG:nmt.model:  Number of Layers: 1
DEBUG:nmt.model:  ===FORWARD PASS===
DEBUG:nmt.model:  ====Running encode()=====
DEBUG:nmt.model:  source_padded shape: torch.Size([19, 2]), source_lengths: [19, 17]
DEBUG:nmt.model:  (1) X shape after embedding: torch.Size([19, 2, 64])
DEBUG:nmt.model:  (2) X shape after post_embed_cnn: torch.Size([19, 2, 64])
DEBUG:nmt.model:  (3) enc_hiddens shape after encoder: torch.Size([19, 2, 256])
DEBUG:nmt.model:  (3) last_hidden shape: torch.Size([2, 2, 128]), last_cell shape: torch.Size([2, 2, 128])
DEBUG:nmt.model:  (3) enc_hiddens shape after permute: torch.Size([2, 19, 256])
DEBUG:nmt.model:  (4) last_hidden shape after concatenation: torch.Size([2, 256])
DEBUG:nmt.model:  (4) last_cell shape after concatenation: torch.Size([2, 256])
DEBUG:nmt.model:  (4) dec_init_state[0] sha

### 03 Forward pass: `decode` method (1)

In [69]:
# Instantiate the model
model = NMT(embed_size=embed_size, 
            hidden_size=hidden_size, 
            vocab=vocab,
            dropout_rate=0.0)

# Forward pass
output = model(src_padded, src_lengths, tgt_padded)

DEBUG:nmt.model:  ===INIT NMT MODEL===
DEBUG:nmt.model:  Embed size: 64, Hidden size: 128, Dropout rate: 0.0
DEBUG:nmt.model:  Source vocab size: 21001, Target vocab size: 8001
DEBUG:nmt.model:  ===FORWARD PASS===
DEBUG:nmt.model:  ---Running encode()---
DEBUG:nmt.model:  ---Running decode()---
DEBUG:nmt.model:  (1a) target_padded shape before removing EOS: torch.Size([34, 2])
DEBUG:nmt.model:  (1a) target_padded shape after removing EOS: torch.Size([33, 2])
DEBUG:nmt.model:  (1b) dec_init_state type: <class 'tuple'>
DEBUG:nmt.model:  (1b) dec_state[0] shape: torch.Size([2, 128])
DEBUG:nmt.model:  (1b) dec_state[1] shape: torch.Size([2, 128])
DEBUG:nmt.model:  (2a) batch_size: 2
DEBUG:nmt.model:  (2b) enc_hiddens shape: torch.Size([2, 19, 256])
DEBUG:nmt.model:  (2b) enc_hiddens_proj shape: torch.Size([2, 19, 128])
DEBUG:nmt.model:  ---Running step()---
DEBUG:nmt.model:  (1b) dec_hidden shape: torch.Size([2, 128]), dec_cell state shape: torch.Size([2, 128])
DEBUG:nmt.model:  (2a) enc_h

### 04 Attention theory

Let's start with the lectures and then look into a more general case in the assignment 3.

We have the following formulas for Luong attention mechanism:

$$
e^t = [s_t^T h_1, ... , s_t^T h_N]
$$

where $s_t$ is the decoder hidden state at time step $t$ and $h_i$ are the encoder hidden states.

$$
\alpha^t = \text{softmax}(e^t)
$$

$$
a_t = \sum_{i=1}^{N} \alpha_i^t h_i
$$


Finally we concatenate the attention output $a_t$ with the decoder hidden
state $s_t$ to form the context vector:

$$
\tilde{s}_t = [a_t; s_t]
$$

---

In assignment 3 the mechanism is slightly different since we use a bidirectional encoder. To compute the dot product between the decoder hidden state $s_t$ and the encoder hidden states $h_i$, we need first to project the encoder hidden states $h_i$ into the same dimensional space as the decoder hidden state $s_t$. 

We also use a slightly different notation:

$$
e^t = [(h_t^{\text{dec}})^T W_{attProj} h_1^{\text{enc}}, ... , (h_t^{\text{dec}})^T W_{attProj} h_N^{\text{enc}}]
$$

This is the same as the following:

$$
e_{t, i} = (h_t^{\text{dec}})^T W_{attProj} h_i^{\text{enc}}
$$

where 
$h_t^{\text{dec}}$ is the decoder hidden state at time step $t$,
$h_i^{\text{enc}}$ are the encoder hidden states, and
$W_{attProj}$ is the projection matrix that maps the *bidirectional* encoder hidden states to the same dimensional space as the decoder hidden state.

The other 2 formulas are exactly the same as in lectures, they just use the new notation:

$$
\alpha^t = \text{softmax}(e^t)
$$

$$
a_t = \sum_{i=1}^{N} \alpha_{t, i} h_i^{\text{enc}}
$$

But then there are some additional steps, not mentioned in the lectures. We now concatenate the attention output $a_t$ with the decoder hidden state $h_t^{\text{dec}}$ and pass this through a linear layer, tanh, and dropout to attain the combined-output vector $o_t$.

$$
u_t = [a_t; h_t^{\text{dec}}]
$$

$$
v_t = W_u u_t
$$

$$
o_t = \text{dropout}(\tanh(v_t))
$$
$$

### 05 `decode` and `step` methods description

#### `decode` method

- In `decode` method we compute the projection of Encoder hidden states `enc_hiddens` using the attention projection layer to obtain `enc_hiddens_proj`. We need to do this only *once* since all the steps of the Decoder will use the *same* projected encoder hidden states. We use the projection for the Encoder is bidirectional while the Decoder is unidirectional.

```python
enc_hiddens_proj = self.att_projection(enc_hiddens)
```

```
DEBUG:nmt.model:  (2b) enc_hiddens shape: torch.Size([2, 19, 256])
DEBUG:nmt.model:  (2b) enc_hiddens_proj shape: torch.Size([2, 19, 128])
```

- We compute a context vector `Ybar_t` of shape `[batch_size, embed_size + hidden_size]` by concatenating:
    - Input embedding of the current timestep `Y_t`, shape `[batch_size, embed_size]` and
    - Combined-output vector `o_prev` from the previous timestamp, shape `[batch_size, hidden_size]`.

```python
Ybar_t = torch.cat([Y_t, o_prev], dim=-1)  # [batch_size, embed_size + hidden_size]
```

Here's a high level pseudocode for `decode` method in the NMT model:

```python
def decode(enc_hiddens, enc_masks, dec_init_state, target_padded):
    
    # (1a) Chop off the <END> token for teacher forcing
    # (2b) Apply the attention projection layer to `enc_hiddens` (used in `step` method)
    # (3a) Construct tensor `Y` of target sentences by applying embedding layer: [tgt_len-1, batch_size, embed_size]
    # (3b) MAIN LOOP. Iterate over the target sequence Y_t one step at a time: [1, batch_size, embed_size]
    #     -  Combine Y_t and o_{t-1} to form Ybar_t
    #     - Pass Ybar_t to `step` method of the decoder. Initially, `dec_state` is the final state  
    #       of the Encoder. Then it is produced by unrolling LSTM Cell for a single step inside `step method`
    #      `step` method also produces o_t for the current time step. 
    # (4) Convert combined_outputs [o_0, o_1, ..., o_{T}] to a single tensor `combined_outputs`.

    # return combined_outputs
```

#### `step` method

In `step` method we do 2 main things:
1. Unroll a single step of the decoder using the previous decoder state and the attention context vector. Just a single line.
2. Compute the attention scores, apply the softmax to obtain the attention distribution, and use it to compute the context vector for the current decoding step. The majority of the method.

Let's look at the unrolling step:

```python
# self.decoder = nn.LSTMCell(input_size=embed_size + hidden_size, hidden_size=hidden_size, bias=True)
# Ybar_t: [batch_size, embed_size + hidden_size]
dec_state = self.decoder(Ybar_t, dec_state)
# dec_state: (dec_hidden, dec_cell) each of shape [batch_size, hidden_size]
dec_hidden, dec_cell = dec_state
```

Let's look at attention computations:

- Compute the attention scores `e_t`. We use PyTorch `bmm` method to perform batch matrix multiplication. We use formula (7) from the assignment description. We compute `enc_hiddens_proj` in `encode` method once as described above.

```python
e_t = torch.bmm(enc_hiddens_proj, dec_hidden.unsqueeze(2)).squeeze(2)  # shape: [batch_size, src_len]
```

- We then apply a mask to `e_t` to ignore the positions corresponding to padding tokens in the source sequence (we set them to `-inf`, so that they will disappear after the softmax). After masking, we apply the softmax function to obtain the attention distribution `alpha_t`. So far a classic attention mechanism.

- Finally, we compute attention output vector, `a_t` (as it is called in assignment notes). In this case we use actual `enc_hiddens`, not the projected version `enc_hiddens_proj`, and perform a batched matrix multiplication with `alpha_t`:

```python
a_t = torch.bmm(alpha_t.unsqueeze(1), enc_hiddens).squeeze(1)  # shape: [batch_size, 2*hidden_size]
```

This is a classic 3-step attention mechanism. 
1. Compute attention scores `e_t`.
2. Apply softmax to obtain attention distribution `alpha_t`.
3. Compute attention output vector `a_t` using `alpha_t` and `enc_hiddens`.

The following steps are seems to be a variation of the classic attention mechanism, where we further process the attention output vector `a_t` - see the formulas for $u_t$, $v_t$, and $o_t$ above. Finally, we obtain the combined output vector `o_t` of shape `[batch_size, hidden_size]`.

### 06 Forward pass: `decode` method (2)

In [56]:
src_padded.shape, src_lengths, tgt_padded.shape

(torch.Size([19, 2]), [19, 17], torch.Size([34, 2]))

In [85]:
# Instantiate the model
model = NMT(embed_size=embed_size, 
            hidden_size=hidden_size, 
            vocab=vocab,
            dropout_rate=0.0)

# Forward pass
output = model(src_padded, src_lengths, tgt_padded)

DEBUG:nmt.model:  ===INIT NMT MODEL===
DEBUG:nmt.model:  Embed size: 64, Hidden size: 128, Dropout rate: 0.0
DEBUG:nmt.model:  Source vocab size: 21001, Target vocab size: 8001
DEBUG:nmt.model:  ===FORWARD PASS===
DEBUG:nmt.model:  ---Running encode()---
DEBUG:nmt.model:  ---Running decode()---
DEBUG:nmt.model:  (1a) target_padded shape before removing EOS: torch.Size([34, 2])
DEBUG:nmt.model:  (1a) target_padded shape after removing EOS: torch.Size([33, 2])
DEBUG:nmt.model:  (1b) dec_init_state type: <class 'tuple'>
DEBUG:nmt.model:  (1b) dec_state[0] shape: torch.Size([2, 128])
DEBUG:nmt.model:  (1b) dec_state[1] shape: torch.Size([2, 128])
DEBUG:nmt.model:  (2a) batch_size: 2
DEBUG:nmt.model:  (2b) enc_hiddens shape: torch.Size([2, 19, 256])
DEBUG:nmt.model:  (2b) enc_hiddens_proj shape: torch.Size([2, 19, 128])
DEBUG:nmt.model:  (3a) target_padded shape: torch.Size([33, 2])
DEBUG:nmt.model:  (3b) Y_t shape before squeeze: torch.Size([1, 2, 64])
DEBUG:nmt.model:  (3b) Y_t shape afte

Let's see how `torch.split` works. It splits a batch of embedded tensors of the shape `[seq_len, batch_size, embedding_dim]` into smaller tensors along the `dim=0` in our case. It returns a tuple of `seq_len` tensors, each of the shape `[1, batch_size, embedding_dim]`.

In [ ]:
target_padded = tgt_padded[:-1]
print(f"target_padded shape: {target_padded.shape}")
Y = model.model_embeddings.target(target_padded)
print(f"Y shape: {Y.shape}")

target_padded shape: torch.Size([33, 2])
Y shape: torch.Size([33, 2, 64])


In [ ]:
type(torch.split(Y, 1, dim=0)), len(torch.split(Y, 1, dim=0)), torch.split(Y, 1, dim=0)[0].shape

(tuple, 33, torch.Size([1, 2, 64]))

### 07 Forward pass: `forward` method

In [22]:
# Instantiate the model
model = NMT(embed_size=embed_size, 
            hidden_size=hidden_size, 
            vocab=vocab,
            dropout_rate=0.0)

# Forward pass
output = model(src_padded, src_lengths, tgt_padded)

DEBUG:nmt.model:  ===INIT NMT MODEL===
DEBUG:nmt.model:  Embed size: 64, Hidden size: 128, Dropout rate: 0.0
DEBUG:nmt.model:  Source vocab size: 21001, Target vocab size: 8001
DEBUG:nmt.model:  ===FORWARD PASS===
DEBUG:nmt.model:  (1) enc_hiddens shape: torch.Size([2, 19, 256])
DEBUG:nmt.model:  (2a) source_lengths: [19, 17]
DEBUG:nmt.model:  (2a) enc_masks shape: torch.Size([2, 19])
DEBUG:nmt.model:  (2a) enc_masks :
tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.,
         1.]])
DEBUG:nmt.model:  (2b) combined_outputs shape: torch.Size([33, 2, 128])
DEBUG:nmt.model:  (3a) logits shape: torch.Size([33, 2, 8001])
DEBUG:nmt.model:  (3b) target_gold shape: torch.Size([33, 2])
DEBUG:nmt.model:  (3d) scores shape: torch.Size([2])
